# PI Few-Shot Raw CSI Prototypical Training

Training-only notebook. It creates a run folder, saves `run.json` before training, writes crash-recovery checkpoints during training, and saves the best model as the canonical `model.pt` used by the evaluation pipeline.

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

RAW_CSI_DIR = PROJECT_ROOT / "data" / "raw_csi_traces_pi"


## Run And Training Configuration

In [ ]:
import gc
import random

import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import clear_output, display
from torch.cuda import is_available
from tqdm.auto import tqdm

from wifi_doppler.data.raw_csi_dataset import RawCsiWindowDataset
from wifi_doppler.evaluation.fewshot import window_true_labels_from_recordings
from wifi_doppler.experiments.artifacts import plot_step_curves, save_checkpoint, save_json
from wifi_doppler.experiments.runs import ensure_run, refresh_run_checkpoint, run_checkpoint_path, run_dir, training_dir
from wifi_doppler.models.raw_csi import RawCsiTemporalEncoder, count_trainable_parameters
from wifi_doppler.training.prototypical import (
    evaluate_cross_dataset_episodes,
    evaluate_same_dataset_episodes,
    load_episode_by_recording,
    prototypical_loss,
    sample_domain_cross_episode_indices,
    sample_episode_indices,
    window_domains_from_recordings,
)

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
device = "cuda" if is_available() else "cpu"

MODEL_RUN_ID = "20260531_raw_csi_proto_domain_separated_v2"
MODEL_LABEL = "Raw CSI proto, domain-separated episodes v2"
MODEL_KEY = "raw_csi_proto"
REPRESENTATION = "raw_csi"
BUILDER = "raw_csi_proto"
TRAINING_OBJECTIVE = "prototypical"
EPISODE_STYLE = "domain-separated"

PERSONS = ("p03", "p05", "p06", "p07", "p08", "p09", "p10", "p11", "p12", "p13")
TRAIN_SCENARIOS = ("PI-1a", "PI-2a", "PI-3a")
TARGET_SCENARIOS = ("PI-4a",)
WINDOW_SIZE = 340
WINDOW_STRIDE = 30
SPLIT_GUARD = 31

SOURCE_TRAIN_SPLIT = (0.0, 0.6)
SOURCE_VAL_SPLIT = (0.6, 0.8)
TARGET_ENROLLMENT_SPLIT = (0.0, 0.6)
TARGET_QUERY_SPLIT = (0.6, 0.8)

TOTAL_PROTO_STEPS = 3000
EVAL_EVERY_STEPS = 100
SAVE_EVERY_STEPS = 25
PROTO_TEST_EPISODES_PER_EVAL = 2
LIVE_PLOT_EVERY_STEPS = 25
PROTO_N_WAY = len(PERSONS)
PROTO_K_SHOT = 5
PROTO_Q_QUERY = 8
PROTO_LR = 1e-4
PROTO_EMBEDDING_DIM = 128
RAW_IN_CHANNELS = 4 * 242
RAW_CHANNEL_MIXER_DIM = 128
RAW_HIDDEN_DIM = 256
PROTO_TEMPERATURE = 0.1
PROTO_EPISODE_STYLE = "domain_cross"
SOURCE_DOMAIN_PAIRS = [(a, b) for a in TRAIN_SCENARIOS for b in TRAIN_SCENARIOS if a != b]
EARLY_STOP_METRIC = "proto_target_val_acc"
EARLY_STOP_PATIENCE_EVALS = 4
EARLY_STOP_MIN_DELTA = 0.0
METRIC = "cosine"

run_path = ensure_run(
    PROJECT_ROOT,
    model_run_id=MODEL_RUN_ID,
    label=MODEL_LABEL,
    model_key=MODEL_KEY,
    representation=REPRESENTATION,
    builder=BUILDER,
    training_objective=TRAINING_OBJECTIVE,
    episode_style=EPISODE_STYLE,
)
train_artifact_dir = training_dir(PROJECT_ROOT, MODEL_RUN_ID)
train_artifact_dir.mkdir(parents=True, exist_ok=True)

print("Run directory:", run_path)
print("Raw CSI directory:", RAW_CSI_DIR)
print("Train scenarios:", TRAIN_SCENARIOS)
print("Target scenarios:", TARGET_SCENARIOS)
print("Labels:", PERSONS)
print("Device:", device)


## Build Datasets

In [ ]:
raw_train_dataset = RawCsiWindowDataset(
    RAW_CSI_DIR,
    scenarios=TRAIN_SCENARIOS,
    split=SOURCE_TRAIN_SPLIT,
    window_size=WINDOW_SIZE,
    window_stride=WINDOW_STRIDE,
    split_guard=SPLIT_GUARD,
    labels=PERSONS,
    flatten_channels=True,
    cache_traces=True,
)

raw_source_val_dataset = RawCsiWindowDataset(
    RAW_CSI_DIR,
    scenarios=TRAIN_SCENARIOS,
    split=SOURCE_VAL_SPLIT,
    window_size=WINDOW_SIZE,
    window_stride=WINDOW_STRIDE,
    split_guard=SPLIT_GUARD,
    labels=PERSONS,
    flatten_channels=True,
    cache_traces=True,
)

raw_target_enrollment_dataset = RawCsiWindowDataset(
    RAW_CSI_DIR,
    scenarios=TARGET_SCENARIOS,
    split=TARGET_ENROLLMENT_SPLIT,
    window_size=WINDOW_SIZE,
    window_stride=WINDOW_STRIDE,
    split_guard=SPLIT_GUARD,
    labels=PERSONS,
    flatten_channels=True,
    cache_traces=True,
)

raw_target_query_dataset = RawCsiWindowDataset(
    RAW_CSI_DIR,
    scenarios=TARGET_SCENARIOS,
    split=TARGET_QUERY_SPLIT,
    window_size=WINDOW_SIZE,
    window_stride=WINDOW_STRIDE,
    split_guard=SPLIT_GUARD,
    labels=PERSONS,
    flatten_channels=True,
    cache_traces=True,
)

raw_train_labels = window_true_labels_from_recordings(raw_train_dataset)
raw_train_domains = window_domains_from_recordings(raw_train_dataset)
raw_source_val_labels = window_true_labels_from_recordings(raw_source_val_dataset)
raw_target_enrollment_labels = window_true_labels_from_recordings(raw_target_enrollment_dataset)
raw_target_query_labels = window_true_labels_from_recordings(raw_target_query_dataset)

print("Raw source train windows:", len(raw_train_dataset))
print("Raw source train domains:", sorted(set(raw_train_domains.tolist())))
print("Raw source validation windows:", len(raw_source_val_dataset))
print("Raw target enrollment windows:", len(raw_target_enrollment_dataset))
print("Raw target query windows:", len(raw_target_query_dataset))
print("Raw sample shape:", tuple(raw_train_dataset[0][0].shape))


## Model And Episode Helpers

In [ ]:
def build_raw_proto_encoder(device: str | torch.device):
    model = RawCsiTemporalEncoder(
        in_channels=RAW_IN_CHANNELS,
        embedding_dim=PROTO_EMBEDDING_DIM,
        channel_mixer_dim=RAW_CHANNEL_MIXER_DIM,
        hidden_dim=RAW_HIDDEN_DIM,
        normalize=True,
    ).to(device)
    with torch.no_grad():
        dummy = raw_target_enrollment_dataset[0][0].unsqueeze(0).to(device)
        _ = model.forward_embedding(dummy)
    print("Raw CSI proto trainable parameters:", count_trainable_parameters(model))
    return model


def eval_source_episodes(model, rng):
    return evaluate_same_dataset_episodes(
        model,
        raw_source_val_dataset,
        raw_source_val_labels,
        device=device,
        n_episodes=PROTO_TEST_EPISODES_PER_EVAL,
        n_way=PROTO_N_WAY,
        k_shot=PROTO_K_SHOT,
        q_query=PROTO_Q_QUERY,
        rng=rng,
        metric=METRIC,
        temperature=PROTO_TEMPERATURE,
        fast_by_recording=True,
    )


def eval_target_episodes(model, rng):
    return evaluate_cross_dataset_episodes(
        model,
        raw_target_enrollment_dataset,
        raw_target_enrollment_labels,
        raw_target_query_dataset,
        raw_target_query_labels,
        device=device,
        n_episodes=PROTO_TEST_EPISODES_PER_EVAL,
        n_way=PROTO_N_WAY,
        k_shot=PROTO_K_SHOT,
        q_query=PROTO_Q_QUERY,
        rng=rng,
        metric=METRIC,
        temperature=PROTO_TEMPERATURE,
        fast_by_recording=True,
    )


def show_live_proto_curves(history):
    fig, _ = plot_step_curves(
        history,
        loss_keys=["proto_train_loss", "proto_source_val_loss", "proto_target_val_loss"],
        acc_keys=["proto_train_acc", "proto_source_val_acc", "proto_target_val_acc"],
        title="Raw CSI prototypical training",
    )
    clear_output(wait=True)
    display(fig)
    plt.close(fig)


## Train

In [ ]:
proto_config = {
    "raw_csi_dir": RAW_CSI_DIR,
    "persons": PERSONS,
    "train_scenarios": TRAIN_SCENARIOS,
    "target_scenarios": TARGET_SCENARIOS,
    "source_train_split": SOURCE_TRAIN_SPLIT,
    "source_val_split": SOURCE_VAL_SPLIT,
    "target_enrollment_split": TARGET_ENROLLMENT_SPLIT,
    "target_query_split": TARGET_QUERY_SPLIT,
    "window_size": WINDOW_SIZE,
    "window_stride": WINDOW_STRIDE,
    "split_guard": SPLIT_GUARD,
    "raw_in_channels": RAW_IN_CHANNELS,
    "raw_channel_mixer_dim": RAW_CHANNEL_MIXER_DIM,
    "raw_hidden_dim": RAW_HIDDEN_DIM,
    "proto_embedding_dim": PROTO_EMBEDDING_DIM,
    "total_proto_steps": TOTAL_PROTO_STEPS,
    "eval_every_steps": EVAL_EVERY_STEPS,
    "save_every_steps": SAVE_EVERY_STEPS,
    "proto_val_episodes_per_eval": PROTO_TEST_EPISODES_PER_EVAL,
    "proto_n_way": PROTO_N_WAY,
    "proto_k_shot": PROTO_K_SHOT,
    "proto_q_query": PROTO_Q_QUERY,
    "proto_lr": PROTO_LR,
    "proto_temperature": PROTO_TEMPERATURE,
    "proto_episode_style": PROTO_EPISODE_STYLE,
    "source_domain_pairs": SOURCE_DOMAIN_PAIRS,
    "early_stop_metric": EARLY_STOP_METRIC,
    "early_stop_patience_evals": EARLY_STOP_PATIENCE_EVALS,
    "early_stop_min_delta": EARLY_STOP_MIN_DELTA,
    "metric": METRIC,
    "seed": SEED,
}


def latest_proto_metrics(history):
    return {
        "step": history["step"][-1],
        "latest_proto_train_loss": history["proto_train_loss"][-1],
        "latest_proto_train_acc": history["proto_train_acc"][-1],
        "latest_proto_source_val_loss": history["proto_source_val_loss"][-1],
        "latest_proto_source_val_acc": history["proto_source_val_acc"][-1],
        "latest_proto_target_val_loss": history["proto_target_val_loss"][-1],
        "latest_proto_target_val_acc": history["proto_target_val_acc"][-1],
    }


def training_payload(history, *, stopped_early=False):
    return {
        "history": history,
        "config": proto_config,
        "selection": {
            "best_metric": EARLY_STOP_METRIC,
            "best_metric_value": best_metric_value,
            "best_metric_step": best_metric_step,
            "stopped_early": stopped_early,
        },
    }


def save_latest_checkpoint(model, history, *, stopped_early=False):
    save_checkpoint(
        model,
        train_artifact_dir,
        labels=PERSONS,
        config=proto_config,
        metrics={**latest_proto_metrics(history), "stopped_early": stopped_early},
        history=history,
        name="latest.pt",
    )
    save_json(train_artifact_dir / "history.json", training_payload(history, stopped_early=stopped_early))


def save_best_checkpoint(model, history):
    save_checkpoint(
        model,
        run_dir(PROJECT_ROOT, MODEL_RUN_ID),
        labels=PERSONS,
        config=proto_config,
        metrics={
            **latest_proto_metrics(history),
            "best_metric": EARLY_STOP_METRIC,
            "best_metric_value": best_metric_value,
            "best_metric_step": best_metric_step,
        },
        history=history,
        name="model.pt",
    )
    refresh_run_checkpoint(PROJECT_ROOT, MODEL_RUN_ID)


def is_better_metric(value, best_value):
    return np.isfinite(value) and value > best_value + EARLY_STOP_MIN_DELTA


proto_model = build_raw_proto_encoder(device)
optimizer = torch.optim.Adam(proto_model.parameters(), lr=PROTO_LR)
episode_rng = np.random.default_rng(SEED)
source_val_episode_rng = np.random.default_rng(SEED + 1)
target_val_episode_rng = np.random.default_rng(SEED + 2)

proto_history = {
    "step": [],
    "proto_train_loss": [],
    "proto_train_acc": [],
    "proto_source_val_loss": [],
    "proto_source_val_acc": [],
    "proto_target_val_loss": [],
    "proto_target_val_acc": [],
}

# Step-0 validation before any optimizer update.
source_val_metrics = eval_source_episodes(proto_model, source_val_episode_rng)
target_val_metrics = eval_target_episodes(proto_model, target_val_episode_rng)
proto_history["step"].append(0)
proto_history["proto_train_loss"].append(np.nan)
proto_history["proto_train_acc"].append(np.nan)
proto_history["proto_source_val_loss"].append(source_val_metrics["loss"])
proto_history["proto_source_val_acc"].append(source_val_metrics["acc"])
proto_history["proto_target_val_loss"].append(target_val_metrics["loss"])
proto_history["proto_target_val_acc"].append(target_val_metrics["acc"])

best_metric_value = proto_history[EARLY_STOP_METRIC][-1]
best_metric_step = 0
evals_without_improvement = 0
stopped_early = False
save_best_checkpoint(proto_model, proto_history)
save_latest_checkpoint(proto_model, proto_history)

print(f"step 0000/{TOTAL_PROTO_STEPS}: source_acc={source_val_metrics['acc']:.4f} target_acc={target_val_metrics['acc']:.4f}")
show_live_proto_curves(proto_history)

progress = tqdm(range(1, TOTAL_PROTO_STEPS + 1), desc="raw CSI proto training steps")
for step in progress:
    proto_model.train()
    if PROTO_EPISODE_STYLE == "domain_cross":
        support_domain, query_domain = SOURCE_DOMAIN_PAIRS[episode_rng.integers(len(SOURCE_DOMAIN_PAIRS))]
        support_indices, query_indices = sample_domain_cross_episode_indices(
            raw_train_labels,
            raw_train_domains,
            support_domain=support_domain,
            query_domain=query_domain,
            n_way=PROTO_N_WAY,
            k_shot=PROTO_K_SHOT,
            q_query=PROTO_Q_QUERY,
            rng=episode_rng,
        )
    elif PROTO_EPISODE_STYLE == "mixed":
        support_domain, query_domain = "mixed", "mixed"
        support_indices, query_indices = sample_episode_indices(
            raw_train_labels,
            n_way=PROTO_N_WAY,
            k_shot=PROTO_K_SHOT,
            q_query=PROTO_Q_QUERY,
            rng=episode_rng,
        )
    else:
        raise ValueError(f"Unknown PROTO_EPISODE_STYLE: {PROTO_EPISODE_STYLE}")

    episode = load_episode_by_recording(raw_train_dataset, support_indices, query_indices)
    optimizer.zero_grad()
    loss, acc = prototypical_loss(
        proto_model,
        episode,
        device=device,
        metric=METRIC,
        temperature=PROTO_TEMPERATURE,
    )
    loss.backward()
    optimizer.step()

    source_val_metrics = {"loss": np.nan, "acc": np.nan}
    target_val_metrics = {"loss": np.nan, "acc": np.nan}
    should_eval = step % EVAL_EVERY_STEPS == 0 or step == TOTAL_PROTO_STEPS
    if should_eval:
        source_val_metrics = eval_source_episodes(proto_model, source_val_episode_rng)
        target_val_metrics = eval_target_episodes(proto_model, target_val_episode_rng)

    proto_history["step"].append(step)
    proto_history["proto_train_loss"].append(loss.item())
    proto_history["proto_train_acc"].append(acc)
    proto_history["proto_source_val_loss"].append(source_val_metrics["loss"])
    proto_history["proto_source_val_acc"].append(source_val_metrics["acc"])
    proto_history["proto_target_val_loss"].append(target_val_metrics["loss"])
    proto_history["proto_target_val_acc"].append(target_val_metrics["acc"])

    progress.set_postfix(
        train_loss=f"{loss.item():.4f}",
        train_acc=f"{acc:.4f}",
        episode=f"{support_domain}->{query_domain}",
        source_acc="nan" if np.isnan(source_val_metrics["acc"]) else f"{source_val_metrics['acc']:.4f}",
        target_acc="nan" if np.isnan(target_val_metrics["acc"]) else f"{target_val_metrics['acc']:.4f}",
    )

    if step % SAVE_EVERY_STEPS == 0 or step == TOTAL_PROTO_STEPS:
        save_latest_checkpoint(proto_model, proto_history, stopped_early=stopped_early)

    if should_eval:
        current_metric_value = proto_history[EARLY_STOP_METRIC][-1]
        if is_better_metric(current_metric_value, best_metric_value):
            best_metric_value = current_metric_value
            best_metric_step = step
            evals_without_improvement = 0
            save_best_checkpoint(proto_model, proto_history)
            print(f"New best {EARLY_STOP_METRIC}: {best_metric_value:.4f} at step {best_metric_step}")
        else:
            evals_without_improvement += 1
            print(f"No {EARLY_STOP_METRIC} improvement for {evals_without_improvement}/{EARLY_STOP_PATIENCE_EVALS} evals")

        if evals_without_improvement >= EARLY_STOP_PATIENCE_EVALS:
            stopped_early = True
            save_latest_checkpoint(proto_model, proto_history, stopped_early=stopped_early)
            print(f"Early stopping at step {step}. Best {EARLY_STOP_METRIC}={best_metric_value:.4f} at step {best_metric_step}.")
            break

    if step % LIVE_PLOT_EVERY_STEPS == 0 or step == TOTAL_PROTO_STEPS:
        show_live_proto_curves(proto_history)
        print(f"step {step:04d}/{TOTAL_PROTO_STEPS}: train_loss={loss.item():.4f} train_acc={acc:.4f} source_acc={source_val_metrics['acc']:.4f} target_acc={target_val_metrics['acc']:.4f}")

save_latest_checkpoint(proto_model, proto_history, stopped_early=stopped_early)
best_checkpoint = torch.load(run_checkpoint_path(PROJECT_ROOT, MODEL_RUN_ID), map_location=device, weights_only=False)
proto_model.load_state_dict(best_checkpoint["model_state_dict"])
proto_model.eval()

for dataset in (raw_train_dataset, raw_source_val_dataset, raw_target_enrollment_dataset, raw_target_query_dataset):
    dataset.clear_cache()
gc.collect()
if is_available():
    torch.cuda.empty_cache()

print(f"Loaded best model from step {best_metric_step}: {run_checkpoint_path(PROJECT_ROOT, MODEL_RUN_ID)}")


## Save Final Training Artifacts

In [ ]:
save_json(train_artifact_dir / "history.json", training_payload(proto_history, stopped_early=stopped_early))
history_fig, _ = plot_step_curves(
    proto_history,
    loss_keys=["proto_train_loss", "proto_source_val_loss", "proto_target_val_loss"],
    acc_keys=["proto_train_acc", "proto_source_val_acc", "proto_target_val_acc"],
    title="Raw CSI prototypical training",
)
history_fig.savefig(train_artifact_dir / "curves.png", dpi=150)
plt.show()

print("Run directory:", run_dir(PROJECT_ROOT, MODEL_RUN_ID))
print("Best model:", run_checkpoint_path(PROJECT_ROOT, MODEL_RUN_ID))
print("Latest checkpoint:", train_artifact_dir / "latest.pt")
print("Training history:", train_artifact_dir / "history.json")
print("Training curves:", train_artifact_dir / "curves.png")


## Next Evaluation Command

In [ ]:
print(f"""python scripts\\evaluate_run.py ^
  --model-run-id {MODEL_RUN_ID} ^
  --model-key {MODEL_KEY} ^
  --compute-umap ^
  --plot-kshot ^
  --plot-umap ^
  --baseline-run-ids raw_csi_mixed_proto doppler_featuremap_proto doppler_softmax_baseline doppler_pooled_head_proto ^
  --umap-baseline-run-ids raw_csi_mixed_proto doppler_featuremap_proto
""")
